# Drone CV — Combined 3-Platform Training (A100)
YOLOv8m at 1280px on VisDrone + Avata 360 perspective crops.

**Dataset:** 7,717 images (6,471 VisDrone + 1,246 Avata 360 dual-fisheye crops)

**Runtime:** ~2h on A100 | **Model:** YOLOv8m (25.9M params)

In [ ]:
#@title 1. Setup — Mount Drive, install, extract data
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

import torch, os, zipfile, shutil, yaml
from pathlib import Path

print('GPU:', torch.cuda.get_device_name(0))
RESULTS_DIR = '/content/drive/MyDrive/DroneCV/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Extract Avata 360 training crops from Drive
zip_path = RESULTS_DIR + '/avata360_training.zip'
assert os.path.exists(zip_path), 'Upload avata360_training.zip to DroneCV/results/ first!'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')
n_avata = len(os.listdir('/content/data/avata360_training/images/train'))
print('Avata 360 crops:', n_avata, 'images')

# Download VisDrone dataset
from ultralytics import YOLO
model = YOLO('yolov8m.pt')
# Quick 1-epoch run to trigger VisDrone download
_ = model.train(data='VisDrone.yaml', epochs=1, imgsz=320, batch=1,
                device=0, project='/tmp/warmup', exist_ok=True, verbose=False)
print('VisDrone downloaded')

# Build combined dataset
combined_img = Path('/content/datasets/combined/images/train')
combined_lbl = Path('/content/datasets/combined/labels/train')
val_img = Path('/content/datasets/combined/images/val')
val_lbl = Path('/content/datasets/combined/labels/val')
for d in [combined_img, combined_lbl, val_img, val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

# Symlink VisDrone train
vd = Path('/content/datasets/VisDrone')
for f in (vd / 'images/train').glob('*.jpg'):
    dst = combined_img / f.name
    if not dst.exists(): os.symlink(f, dst)
for f in (vd / 'labels/train').glob('*.txt'):
    dst = combined_lbl / f.name
    if not dst.exists(): os.symlink(f, dst)

# Copy Avata 360 crops
for f in Path('/content/data/avata360_training/images/train').glob('*.jpg'):
    shutil.copy(f, combined_img / f.name)
for f in Path('/content/data/avata360_training/labels/train').glob('*.txt'):
    shutil.copy(f, combined_lbl / f.name)

# Symlink VisDrone val
for f in (vd / 'images/val').glob('*.jpg'):
    dst = val_img / f.name
    if not dst.exists(): os.symlink(f, dst)
for f in (vd / 'labels/val').glob('*.txt'):
    dst = val_lbl / f.name
    if not dst.exists(): os.symlink(f, dst)

# Write YAML
cfg = {'path': '/content/datasets/combined', 'train': 'images/train',
       'val': 'images/val', 'nc': 10,
       'names': ['pedestrian','people','bicycle','car','van',
                 'truck','tricycle','awning-tricycle','bus','motor']}
with open('/content/datasets/combined.yaml', 'w') as f:
    yaml.dump(cfg, f)

n_train = len(list(combined_img.glob('*')))
n_val = len(list(val_img.glob('*')))
print(f'Combined dataset: {n_train} train, {n_val} val')
print(f'  VisDrone: 6471 + Avata 360: {n_avata} = {n_train}')

In [ ]:
#@title 2. Train YOLOv8m — Combined Dataset (~2h on A100)
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
results = model.train(
    data='/content/datasets/combined.yaml',
    epochs=30,
    imgsz=1280,
    batch=8,
    device=0,
    project='/content/runs',
    name='combined_v8m_1280',
    exist_ok=True,
    cos_lr=True,
    patience=10,
    plots=True,
)
print('Training complete!')

In [ ]:
#@title 3. Save to Drive
from pathlib import Path
import shutil

src = Path('/content/runs/combined_v8m_1280/weights/best.pt')
dst = '/content/drive/MyDrive/DroneCV/results/combined_v8m_1280_best.pt'
shutil.copy(src, dst)
print(f'Saved: {dst} ({src.stat().st_size / 1e6:.1f} MB)')